# Bioinformatics & Data Science: take-home toolkit
**Liceo Scientifico "Enzo Ferrari", Cesenatico**, Martina Castellucci

All the code from the two sessions, ready to run and change.

**How to use it**
1. In Google Colab: *File > Save a copy in Drive*, so you can edit your own copy.
2. Run the cells in order with **Shift + Enter**.
3. Change the values, break things, run again. Nothing can go wrong.

**Contents** (only what we did in class)
1. DNA as text
2. From DNA to protein
3. Mutations
4. Needleman-Wunsch alignment
5. A decision tree learns a rule
6. Recall, precision and thresholds
7. SQL: counting signed reports
8. Exercises and solutions

## 1. DNA is text
A DNA sequence is a string: Python can measure it, count letters and compute the GC content.

In [1]:
dna = "ATGGTGCACCTGACTCCTGAGGAGAAG"   # start of the human beta-globin gene (HBB)

print("Length:", len(dna))
for base in "ACGT":
    print(base, dna.count(base))

def gc_content(seq):
    gc = seq.count("G") + seq.count("C")
    return gc / len(seq)

print("GC content:", round(gc_content(dna), 2))

Length: 27
A 7
C 6
G 9
T 5
GC content: 0.56


## 2. From DNA to protein
The genetic code is a lookup table: 64 codons, 20 amino acids and 3 stop signals (`*`). Each line of `amino` is one block of the textbook table (first base T, C, A, G).

In [3]:
bases = "TCAG"
amino = ("FFLLSSSSYY**CC*W"
         "LLLLPPPPHHQQRRRR"
         "IIIMTTTTNNKKSSRR"
         "VVVVAAAADDEEGGGG")
codons = [a + b + c for a in bases for b in bases for c in bases]
code = dict(zip(codons, amino))

def translate(seq):
    protein = ""
    for i in range(0, len(seq) - 2, 3):
        protein += code[seq[i:i + 3]]
    return protein

print(len(code), "codons")
print(translate(dna))

64 codons
MVHLTPEEK


## 3. Mutations
One letter changes, the protein changes. Codon 7 `GAG` becomes `GTG`: glutamic acid (E) becomes valine (V). This is the variant behind sickle cell disease.

In [4]:
def mutate(seq, position, new_base):
    return seq[:position] + new_base + seq[position + 1:]

sickle = mutate(dna, 19, "T")

print(translate(dna), "healthy")
print(translate(sickle), "sickle cell")

# Try a synonymous mutation: change the third letter of codon 2 (GTG -> GTA)
silent = mutate(dna, 5, "A")
print(translate(silent), "same protein?", translate(silent) == translate(dna))

MVHLTPEEK healthy
MVHLTPVEK sickle cell
MVHLTPEEK same protein? True


## 4. Needleman-Wunsch alignment
The grid from Challenge A, filled by the computer. Match +1, mismatch -1, gap -2. The function returns the grid, the best score and one best alignment.

In [5]:
def needleman_wunsch(a, b, match=1, mismatch=-1, gap=-2):
    n, m = len(a), len(b)
    F = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        F[i][0] = i * gap
    for j in range(m + 1):
        F[0][j] = j * gap
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            s = match if a[i - 1] == b[j - 1] else mismatch
            F[i][j] = max(F[i - 1][j - 1] + s, F[i - 1][j] + gap, F[i][j - 1] + gap)
    # traceback from the bottom-right cell
    top, bottom, i, j = "", "", n, m
    while i > 0 or j > 0:
        s = match if i > 0 and j > 0 and a[i - 1] == b[j - 1] else mismatch
        if i > 0 and j > 0 and F[i][j] == F[i - 1][j - 1] + s:
            top, bottom, i, j = a[i - 1] + top, b[j - 1] + bottom, i - 1, j - 1
        elif i > 0 and F[i][j] == F[i - 1][j] + gap:
            top, bottom, i = a[i - 1] + top, "-" + bottom, i - 1
        else:
            top, bottom, j = "-" + top, b[j - 1] + bottom, j - 1
    return F, F[n][m], top, bottom

def show(a, b):
    F, score, top, bottom = needleman_wunsch(a, b)
    print("     " + "".join(f"{c:>4}" for c in "-" + b))
    for label, row in zip("-" + a, F):
        print(f"{label:>4} " + "".join(f"{v:>4}" for v in row))
    print("\nScore:", score)
    print(top)
    print(bottom)

show("GATC", "GAC")
print()
show("GAATTC", "GATTA")

        -   G   A   C
   -    0  -2  -4  -6
   G   -2   1  -1  -3
   A   -4  -1   2   0
   T   -6  -3   0   1
   C   -8  -5  -2   1

Score: 1
GATC
GA-C

        -   G   A   T   T   A
   -    0  -2  -4  -6  -8 -10
   G   -2   1  -1  -3  -5  -7
   A   -4  -1   2   0  -2  -4
   A   -6  -3   0   1  -1  -1
   T   -8  -5  -2   1   2   0
   T  -10  -7  -4  -1   2   1
   C  -12  -9  -6  -3   0   1

Score: 1
GAATTC
G-ATTA


## 5. A decision tree learns a rule
The made-up data from Challenge C. With the column `retested` the model cheats (data leakage). Without it, it finds the rule on its own.

In [8]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

train = pd.DataFrame({
    "food":      ["Maize", "Maize", "Maize", "Maize", "Rice", "Rice", "Swordfish", "Swordfish", "Apple", "Apple", "Maize", "Rice"],
    "origin":    ["non-EU", "non-EU", "EU", "EU", "non-EU", "EU", "non-EU", "EU", "EU", "non-EU", "non-EU", "non-EU"],
    "season":    ["summer", "winter", "summer", "winter", "summer", "winter", "summer", "winter", "summer", "winter", "summer", "winter"],
    "retested":  ["yes", "yes", "no", "no", "no", "no", "yes", "yes", "no", "no", "yes", "no"],
    "non_compliant": ["yes", "yes", "no", "no", "no", "no", "yes", "yes", "no", "no", "yes", "no"],
})
test = pd.DataFrame({
    "food":   ["Maize", "Swordfish", "Rice", "Maize", "Apple", "Maize"],
    "origin": ["non-EU", "EU", "EU", "EU", "non-EU", "non-EU"],
    "season": ["summer", "summer", "summer", "summer", "summer", "winter"],
    "non_compliant": ["yes", "yes", "no", "yes", "no", "no"],
})
y = train["non_compliant"]

for cols in (["food", "origin", "season", "retested"], ["food", "origin", "season"]):
    X = pd.get_dummies(train[cols])
    model = DecisionTreeClassifier(random_state=0).fit(X, y)
    print("Columns:", cols)
    print(export_text(model, feature_names=list(X.columns)))

X_test = pd.get_dummies(test[cols]).reindex(columns=X.columns, fill_value=False)
test["prediction"] = model.predict(X_test)
print(test)
print("Correct:", (test["prediction"] == test["non_compliant"]).sum(), "out of", len(test))

Columns: ['food', 'origin', 'season', 'retested']
|--- retested_yes <= 0.50
|   |--- class: no
|--- retested_yes >  0.50
|   |--- class: yes

Columns: ['food', 'origin', 'season']
|--- food_Swordfish <= 0.50
|   |--- food_Maize <= 0.50
|   |   |--- class: no
|   |--- food_Maize >  0.50
|   |   |--- origin_EU <= 0.50
|   |   |   |--- class: yes
|   |   |--- origin_EU >  0.50
|   |   |   |--- class: no
|--- food_Swordfish >  0.50
|   |--- class: yes

        food  origin  season non_compliant prediction
0      Maize  non-EU  summer           yes        yes
1  Swordfish      EU  summer           yes        yes
2       Rice      EU  summer            no         no
3      Maize      EU  summer           yes         no
4      Apple  non-EU  summer            no         no
5      Maize  non-EU  winter            no        yes
Correct: 4 out of 6


## 6. Recall, precision and thresholds
The 20 batches from Challenge D. For each threshold we count checks, missed risky batches and total cost.

In [9]:
probability = [0.92, 0.85, 0.78, 0.71, 0.66, 0.61, 0.55, 0.52, 0.47, 0.44,
               0.38, 0.35, 0.31, 0.27, 0.22, 0.18, 0.14, 0.11, 0.07, 0.04]
risky = [1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
CHECK, MISSED = 200, 5000   # euro

for threshold in [0.3, 0.4, 0.5, 0.7]:
    checked = [p >= threshold for p in probability]
    tp = sum(c and r for c, r in zip(checked, risky))
    missed = sum(risky) - tp
    n_checks = sum(checked)
    cost = n_checks * CHECK + missed * MISSED
    print(f"threshold {threshold}: {n_checks:2} checks, recall {tp / sum(risky):.0%}, "
          f"precision {tp / n_checks:.0%}, missed {missed}, cost {cost} euro")

threshold 0.3: 13 checks, recall 86%, precision 46%, missed 1, cost 7600 euro
threshold 0.4: 10 checks, recall 86%, precision 60%, missed 1, cost 7000 euro
threshold 0.5:  8 checks, recall 71%, precision 62%, missed 2, cost 11600 euro
threshold 0.7:  4 checks, recall 43%, precision 75%, missed 4, cost 20800 euro


## 7. SQL: counting signed reports
The query from slide 36, on the same made-up data: hospital A signs 90 reports out of 100, hospital B 450 out of 900. Together, is it 70%?

**Exercise D.** Write a query for Puglia only: how many reports per area, and what share is signed? Hint: `WHERE h.region = 'Puglia'` and `GROUP BY r.area`.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")   # a small database that lives only in memory
db.executescript("""
CREATE TABLE hospitals (id INTEGER PRIMARY KEY, name TEXT, region TEXT);
CREATE TABLE reports   (id INTEGER PRIMARY KEY, hospital_id INTEGER, area TEXT, signed INTEGER);
""")
db.executemany("INSERT INTO hospitals VALUES (?, ?, ?)",
               [(1, "Hospital A", "Emilia-Romagna"), (2, "Hospital B", "Puglia")])

# Made-up data. Hospital A: 100 reports, 90 signed. Hospital B: 900 reports, 450 signed.
reports = []
for i in range(100):
    reports.append((1, "Radiology" if i < 40 else "Lab", 1 if i < 90 else 0))
for i in range(900):
    reports.append((2, "Radiology" if i < 300 else "Lab", 1 if i < 450 else 0))
db.executemany("INSERT INTO reports (hospital_id, area, signed) VALUES (?, ?, ?)", reports)

query = """
SELECT h.name,
       COUNT(*)                                   AS reports,
       SUM(r.signed)                              AS signed,
       ROUND(100.0 * SUM(r.signed) / COUNT(*), 1) AS signed_pct
FROM reports r
JOIN hospitals h ON r.hospital_id = h.id
GROUP BY h.name;
"""
rows = db.execute(query).fetchall()
for row in rows:
    print(row)

# The trap: the average of the two percentages is not the share for the whole group
pcts = [row[3] for row in rows]
print("Average of the percentages:", sum(pcts) / len(pcts), "%")
total = db.execute("SELECT ROUND(100.0 * SUM(signed) / COUNT(*), 1) FROM reports").fetchone()[0]
print("Real share for the group:", total, "%")

## 8. Exercises
Try them first. Solutions are at the bottom.

**A.** Write `at_content(seq)`, the share of A and T. Check that `gc_content(dna) + at_content(dna)` is 1.

**B.** Find the position of the first stop codon (`*`) in `translate("ATGAAATAGGGC")`.

**C.** Change the costs in section 6: if a missed batch costs only 500 euro, which threshold is cheapest?

In [10]:
# Your code here


### Solutions

In [11]:
# A
def at_content(seq):
    return (seq.count("A") + seq.count("T")) / len(seq)
print(round(gc_content(dna) + at_content(dna), 2))

# B
protein = translate("ATGAAATAGGGC")
print(protein, "first stop at position", protein.index("*"))

# C
for threshold in [0.3, 0.4, 0.5, 0.7]:
    checked = [p >= threshold for p in probability]
    missed = sum(r and not c for c, r in zip(checked, risky))
    print(threshold, sum(checked) * 200 + missed * 500, "euro")

1.0
MK*G first stop at position 2
0.3 3100 euro
0.4 2500 euro
0.5 2600 euro
0.7 2800 euro


In [ ]:
# D
for row in db.execute("""
    SELECT r.area,
           COUNT(*)                                   AS reports,
           ROUND(100.0 * SUM(r.signed) / COUNT(*), 1) AS signed_pct
    FROM reports r
    JOIN hospitals h ON r.hospital_id = h.id
    WHERE h.region = 'Puglia'
    GROUP BY r.area;"""):
    print(row)

---
More to explore: [rosalind.info](https://rosalind.info) (bioinformatics problems), [UniProt](https://www.uniprot.org), [BLAST](https://blast.ncbi.nlm.nih.gov), [AlphaFold DB](https://alphafold.ebi.ac.uk)